# Лабораторная работа 8

**Тема:** поиск экстремума функции одной переменной прямыми методами.

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

**Вариант 6:** найти **максимум** $\;f(x) = x^2 \cos(2x + 1)$

**Задание:**
1. Написать программу поиска экстремума методами дихотомии, золотого сечения и Фибоначчи.
   Обозначения: $x_L, x_R$ — левая и правая точки, $y_L, y_R$ — значения функции в них.
2. Вывести график функции и определить интервал унимодальности $[a, b]$.
3. Входные параметры: концы $[a, b]$, точность $\varepsilon$. Для Фибоначчи задаётся число $n$
   и вычисляется получаемая точность $\varepsilon$.
4. Выходные данные: число итераций, точка экстремума $x_m$ и значение $y_m = f(x_m)$.

## Импорт и параметры

* $\varepsilon$ — допустимая погрешность точки экстремума: $|x_m - x^*| \le \varepsilon$. Поиск идёт,
  пока длина интервала неопределённости больше $2\varepsilon$. Результат — его середина.
* $\delta$ — константа различимости (5). Это наименьший сдвиг по $x$, при котором разница значений $f$ ещё
  надёжно видна. Должно выполняться $\delta < 2\varepsilon$, иначе интервал нельзя сузить до нужной длины.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 100

In [ ]:
ASK_INPUT = False   # True — ввод параметров с клавиатуры при запуске


def ask(prompt, default, cast=float):
    if not ASK_INPUT:
        return default
    s = input(f"{prompt} [{default}]: ").strip()
    return cast(s) if s else default


a = ask("Левый конец a", 1.5)
b = ask("Правый конец b", 4.0)
EPS = ask("Точность ε", 1e-5)
DELTA = ask("Константа различимости δ", 1e-6)
N_FIB = ask("Число вычислений функции n для Фибоначчи", 20, int)

assert DELTA < 2 * EPS, "δ должна быть меньше 2ε (условие (5))"


def f(x):
    return x ** 2 * math.cos(2 * x + 1)


print(f"f(x) = x²·cos(2x + 1), ищем максимум на [{a}, {b}]")
print(f"ε = {EPS},  δ = {DELTA},  n (Фибоначчи) = {N_FIB}")

## 1. График функции и интервал унимодальности

Прямые методы находят экстремум только **унимодальной** функции, то есть функции с одним экстремумом на отрезке.
Функция $x^2\cos(2x + 1)$ колеблется, а её амплитуда растёт как $x^2$. Поэтому у неё много локальных максимумов,
а глобального максимума на всей оси нет. Найдём все экстремумы на $[-4;\ 4]$ по смене знака приращения
и выберем отрезок, на котором максимум ровно один.

In [ ]:
xs = np.linspace(-4, 4, 8001)
ys = xs ** 2 * np.cos(2 * xs + 1)
turn = np.diff(np.sign(np.diff(ys)))

print(f"{'Тип':<10}{'x':>9}{'f(x)':>10}")
for i in np.where(turn != 0)[0]:
    kind = "максимум" if turn[i] < 0 else "минимум"
    print(f"{kind:<10}{xs[i + 1]:>9.3f}{ys[i + 1]:>10.4f}")

seg = (xs >= a) & (xs <= b)
n_max_in_seg = np.sum(turn[seg[1:-1]] < 0)
print(f"\nНа [{a}; {b}] локальных максимумов: {n_max_in_seg} → функция унимодальна")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(xs, ys, "b", lw=2, label="f(x) = x²·cos(2x + 1)")
ax1.axvspan(a, b, color="g", alpha=0.15, label=f"интервал унимодальности [{a}; {b}]")
ax1.axhline(0, color="k", lw=0.8)
ax1.set_title("Функция на [−4; 4]: несколько локальных максимумов")
ax1.set_xlabel("x")
ax1.set_ylabel("f(x)")
ax1.grid(alpha=0.4)
ax1.legend()

xz = np.linspace(a, b, 500)
ax2.plot(xz, xz ** 2 * np.cos(2 * xz + 1), "b", lw=2)
ax2.set_title(f"Отрезок [{a}; {b}]: один максимум")
ax2.set_xlabel("x")
ax2.grid(alpha=0.4)
plt.tight_layout()
plt.show()

### Эталонное значение

Чтобы оценить точность прямых методов, найдём максимум ещё и аналитически. Из $f'(x) = 0$:

$$f'(x) = 2x\cos(2x + 1) - 2x^2\sin(2x + 1) = 0 \;\Rightarrow\; x\,\operatorname{tg}(2x + 1) = 1 \quad (x \ne 0).$$

Это нелинейное уравнение. Решим его методом Ньютона из лабораторной работы 7.

In [ ]:
def g(x):                       # f'(x) / 2x
    return math.cos(2 * x + 1) - x * math.sin(2 * x + 1)


def dg(x):
    return -3 * math.sin(2 * x + 1) - 2 * x * math.cos(2 * x + 1)


x_star = 2.8
for _ in range(50):
    x_star -= g(x_star) / dg(x_star)
y_star = f(x_star)

print(f"Эталон: x* = {x_star:.12f},  f(x*) = {y_star:.12f}")

## 2. Методы последовательного поиска

Все три метода используют **процедуру исключения отрезка**. Для максимума при $x_L < x_R$:
* если $y_L > y_R$, максимум лежит в $[a, x_R]$, и отрезок $[x_R, b]$ отбрасывается;
* если $y_L < y_R$, максимум лежит в $[x_L, b]$, и отрезок $[a, x_L]$ отбрасывается;
* если $y_L = y_R$, максимум лежит в $[x_L, x_R]$.

Функции написаны для максимума и минимума сразу: при поиске минимума сравнение идёт для $-f$.

### Дихотомия

Точки ставятся симметрично середине на расстоянии $\delta/2$ от неё, формула (6):
$x_L = \frac{a + b}{2} - \frac{\delta}{2}$, $\;x_R = \frac{a + b}{2} + \frac{\delta}{2}$.
За шаг интервал уменьшается почти вдвое. На каждом шаге нужны **два** новых вычисления $f$.

### Золотое сечение

Точки (9): $x_L = b - \tau(b - a)$, $\;x_R = a + \tau(b - a)$, где $\tau = 1/\Phi = 0.618\ldots$
Одна точка переходит на следующий шаг, поэтому за шаг нужно **одно** новое вычисление, а интервал сжимается в $\Phi \approx 1.618$ раза.

### Метод Фибоначчи

Число вычислений $n$ задаётся заранее. Точки (11):
$x_L = a + \dfrac{F_{n-i-1}}{F_{n-i+1}}(b - a)$, $\;x_R = a + \dfrac{F_{n-i}}{F_{n-i+1}}(b - a)$.
На последнем шаге обе точки попадают в середину, поэтому одна из них сдвигается на $\delta$.
После $n$ вычислений интервал равен $(b - a)/F_n$ (с точностью до $\delta$). Значит, получаемая точность

$$\varepsilon_n = \frac{b - a}{2F_n} + \frac{\delta}{2}.$$

In [ ]:
TAU = (math.sqrt(5) - 1) / 2       # 1/Φ = 0.618…


def better(y1, y2, mode):
    # True, если y1 «лучше» y2 (больше для max, меньше для min)
    return y1 > y2 if mode == "max" else y1 < y2


def dichotomy(f, a, b, eps, delta, mode="max"):
    hist = [(a, b)]
    evals = 0
    while b - a > 2 * eps:
        mid = (a + b) / 2
        xL, xR = mid - delta / 2, mid + delta / 2     # (6)
        yL, yR = f(xL), f(xR)
        evals += 2
        if better(yL, yR, mode):
            b = xR
        else:
            a = xL
        hist.append((a, b))
    xm = (a + b) / 2
    return xm, f(xm), len(hist) - 1, evals, hist


def golden_section(f, a, b, eps, mode="max"):
    xL, xR = b - TAU * (b - a), a + TAU * (b - a)     # (9)
    yL, yR = f(xL), f(xR)
    evals = 2
    hist = [(a, b)]
    while b - a > 2 * eps:
        if better(yL, yR, mode):
            b, xR, yR = xR, xL, yL
            xL = b - TAU * (b - a)
            yL = f(xL)
        else:
            a, xL, yL = xL, xR, yR
            xR = a + TAU * (b - a)
            yR = f(xR)
        evals += 1
        hist.append((a, b))
    xm = (a + b) / 2
    return xm, f(xm), len(hist) - 1, evals, hist


def fib_numbers(n):
    F = [1, 1]                                        # (10): F0 = F1 = 1
    while len(F) <= n:
        F.append(F[-1] + F[-2])
    return F


def fibonacci(f, a, b, n, delta, mode="max"):
    F = fib_numbers(n)
    xL = a + F[n - 2] / F[n] * (b - a)                # (11)
    xR = a + F[n - 1] / F[n] * (b - a)
    yL, yR = f(xL), f(xR)
    evals = 2
    hist = [(a, b)]
    for k in range(1, n - 1):
        last = k == n - 2                             # на последнем шаге точки совпадают
        if better(yL, yR, mode):
            b, xR, yR = xR, xL, yL
            xL = a + F[n - k - 2] / F[n - k] * (b - a)
            if last:
                xL -= delta
            yL = f(xL)
        else:
            a, xL, yL = xL, xR, yR
            xR = a + F[n - k - 1] / F[n - k] * (b - a)
            if last:
                xR += delta
            yR = f(xR)
        evals += 1
        hist.append((a, b))
    if better(yL, yR, mode):                          # завершающее исключение отрезка
        b = xR
    else:
        a = xL
    hist.append((a, b))
    xm = (a + b) / 2
    return xm, f(xm), len(hist) - 1, evals, hist


def fib_eps(a, b, n, delta):
    return (b - a) / (2 * fib_numbers(n)[n]) + delta / 2


def fib_n_for_eps(a, b, eps, delta):
    n = 3
    while fib_eps(a, b, n, delta) > eps:
        n += 1
    return n

## 3. Тест на примере из теории

В методичке разобран пример: минимум функции $y = x^3 - x + e^{-x}$ на $[0, 1]$.
Пассивным поиском с $\varepsilon = 0.2$ получено $x_m = 0.7 \pm 0.1$. Точное значение из $f'(x) = 3x^2 - 1 - e^{-x} = 0$:
$x^* \approx 0.7056$. Проверим, что все три метода (в режиме минимума) его находят.

In [ ]:
def f_test(x):
    return x ** 3 - x + math.exp(-x)


x_t = 0.7
for _ in range(50):
    x_t -= (3 * x_t ** 2 - 1 - math.exp(-x_t)) / (6 * x_t + math.exp(-x_t))

n_t = fib_n_for_eps(0, 1, EPS, DELTA)
tests = [
    ("Дихотомия", dichotomy(f_test, 0, 1, EPS, DELTA, "min")),
    ("Золотое сечение", golden_section(f_test, 0, 1, EPS, "min")),
    (f"Фибоначчи (n = {n_t})", fibonacci(f_test, 0, 1, n_t, DELTA, "min")),
]
print(f"Тест: минимум x³ − x + e^(−x) на [0, 1],  точное x* = {x_t:.8f}\n")
print(f"{'Метод':<22}{'x_m':>12}{'y_m':>12}{'|x_m − x*|':>13}")
for name, (xm, ym, _, _, _) in tests:
    print(f"{name:<22}{xm:>12.8f}{ym:>12.8f}{abs(xm - x_t):>13.2e}")

## 4. Поиск максимума для варианта 6

In [ ]:
x_d, y_d, it_d, ev_d, h_d = dichotomy(f, a, b, EPS, DELTA)
x_g, y_g, it_g, ev_g, h_g = golden_section(f, a, b, EPS)

n_need = fib_n_for_eps(a, b, EPS, DELTA)
x_f, y_f, it_f, ev_f, h_f = fibonacci(f, a, b, n_need, DELTA)
x_fn, y_fn, it_fn, ev_fn, h_fn = fibonacci(f, a, b, N_FIB, DELTA)

rows = [
    ("Дихотомия", x_d, y_d, it_d, ev_d, (h_d[-1][1] - h_d[-1][0]) / 2),
    ("Золотое сечение", x_g, y_g, it_g, ev_g, (h_g[-1][1] - h_g[-1][0]) / 2),
    (f"Фибоначчи, n = {n_need}", x_f, y_f, it_f, ev_f, fib_eps(a, b, n_need, DELTA)),
    (f"Фибоначчи, n = {N_FIB}", x_fn, y_fn, it_fn, ev_fn, fib_eps(a, b, N_FIB, DELTA)),
]

print(f"Максимум f(x) = x²·cos(2x + 1) на [{a}; {b}],  ε = {EPS}\n")
print(f"{'Метод':<22}{'x_m':>13}{'y_m':>13}{'итер.':>7}{'выч. f':>8}{'точность':>11}{'|x_m − x*|':>12}")
print("-" * 86)
for name, xm, ym, it, ev, acc in rows:
    print(f"{name:<22}{xm:>13.8f}{ym:>13.8f}{it:>7}{ev:>8}{acc:>11.2e}{abs(xm - x_star):>12.2e}")
print(f"{'Эталон (Ньютон)':<22}{x_star:>13.8f}{y_star:>13.8f}")

print(f"\nФибоначчи: для ε = {EPS} нужно n = {n_need} вычислений (F_n = {fib_numbers(n_need)[n_need]})")
print(f"Фибоначчи: при заданном n = {N_FIB} получаемая точность ε = (b − a)/(2F_n) + δ/2 = {fib_eps(a, b, N_FIB, DELTA):.2e}")
print(f"Пассивный поиск для той же ε потребовал бы ≈ (b − a)/ε = {math.ceil((b - a) / EPS)} вычислений")

## 5. Сравнение скорости сходимости

Эффективность прямого метода измеряется тем, как быстро уменьшается интервал неопределённости
с ростом числа вычислений функции. Сравним фактические данные с таблицей 1 методички.

In [ ]:
def lengths_by_evals(hist, first_evals, per_step):
    ev = [0] + [first_evals + per_step * i for i in range(len(hist) - 1)]
    return ev, [bb - aa for aa, bb in hist]


ev_d_list, L_d = lengths_by_evals(h_d, 2, 2)
ev_g_list, L_g = lengths_by_evals(h_g, 2, 1)
ev_f_list = [0] + list(range(2, 2 + len(h_f) - 1))
L_f = [bb - aa for aa, bb in h_f]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(ev_d_list, L_d, "o-", ms=3, label="Дихотомия")
ax1.semilogy(ev_g_list, L_g, "s-", ms=3, label="Золотое сечение")
ax1.semilogy(ev_f_list, L_f, "^-", ms=3, label=f"Фибоначчи, n = {n_need}")
ax1.axhline(2 * EPS, color="r", ls="--", label="2ε")
ax1.set_title("Длина интервала неопределённости")
ax1.set_xlabel("число вычислений f(x)")
ax1.set_ylabel("b − a")
ax1.grid(alpha=0.4, which="both")
ax1.legend()

xz = np.linspace(a, b, 500)
ax2.plot(xz, xz ** 2 * np.cos(2 * xz + 1), "b", lw=2, label="f(x)")
for (aa, bb), c in zip(h_g[:8], plt.cm.viridis(np.linspace(0, 0.9, 8))):
    ax2.plot([aa, bb], [f(x_star) + 0.3] * 2, color=c, lw=6, alpha=0.6)
ax2.plot(x_g, y_g, "r*", ms=15, label=f"x_m = {x_g:.5f}, y_m = {y_g:.5f}")
ax2.set_title("Золотое сечение: первые 8 интервалов неопределённости")
ax2.set_xlabel("x")
ax2.grid(alpha=0.4)
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()

print("Сокращение интервала L1/Ln в зависимости от числа опытов n (как в таблице 1):")
print(f"{'n':>4}{'дихотомия':>12}{'золотое':>12}{'Фибоначчи':>12}")
for n in range(2, 16):
    dich = 2 ** (n // 2)
    gold = (1 / TAU) ** (n - 1)
    fib = fib_numbers(n)[n]
    print(f"{n:>4}{dich:>12}{gold:>12.3f}{fib:>12}")

В таблице 1 методички есть опечатки: для $n = 13$ у дихотомии должно быть 64, а не 63, для $n = 23$ — 2048, а не 2038.

## Вывод

1. Функция $x^2\cos(2x + 1)$ не унимодальна на всей оси: на $[-4;\ 4]$ у неё четыре локальных максимума,
   глобального максимума нет. Для прямых методов выбран интервал унимодальности $[1.5;\ 4]$.
   Максимум на нём: $x_m \approx 2.8124$, $y_m \approx 7.4525$. Совпадает с решением уравнения $x\,\operatorname{tg}(2x + 1) = 1$ методом Ньютона.
2. Все три метода нашли максимум с заданной точностью $\varepsilon = 10^{-5}$. Тест на примере из
   методички (минимум $x^3 - x + e^{-x}$) также пройден.
3. **Дихотомия** сжимает интервал почти вдвое за шаг, но тратит два вычисления функции на шаг.
   **Золотое сечение** переносит одну точку на следующий шаг, поэтому тратит одно вычисление и сжимает
   интервал в 1.618 раза. Для той же точности ему нужно меньше вычислений.
4. **Метод Фибоначчи** — самый экономный при заранее заданном числе вычислений $n$: интервал сокращается
   в $F_n$ раз. Точность выражается через $n$ формулой $\varepsilon_n = (b - a)/(2F_n) + \delta/2$. Недостаток — $n$ нужно знать заранее.
5. Константа различимости $\delta$ ограничивает достижимую точность (5): интервал нельзя сузить меньше $\delta$.